In [1]:
import cobra

In [2]:
model = cobra.io.read_sbml_model("model.xml")

In [3]:
model.reactions.rxn01517_c0

Reaction identifier,rxn01517_c0
Name,ATP:dUMP phosphotransferase [c0]
Memory address,0x15912e350
Stoichiometry,cpd00002_c0 + cpd00067_c0 + cpd00299_c0 <=> cpd00008_c0 + cpd00978_c0 ATP + H+ + dUMP <=> ADP + dUDP
GPR,WP_049588748.1
Lower bound,-1000.0
Upper bound,1000.0


In [4]:
atp_consuming_reactions = [
    "rxn00077_c0",
    "rxn00104_c0",
    "rxn00239_c0",
    "rxn00364_c0",
    "rxn00379_c0",
    "rxn01219_c0",
    "rxn01509_c0",
    "rxn01517_c0",
    "rxn02314_c0",
    "rxn08762_c0",
    "rxn15121_c0",
]

In [5]:
# Make a copy of the original model to modify
amac_model_strict_atp = model.copy()
# Loop through and set all lower bounds to 0
for rxn_id in atp_consuming_reactions:
    if (rxn_id in amac_model_strict_atp.reactions) & (
        amac_model_strict_atp.metabolites.cpd00002_c0
        in amac_model_strict_atp.reactions.get_by_id(rxn_id).reactants
    ):
        amac_model_strict_atp.reactions.get_by_id(rxn_id).lower_bound = 0
    else:
        print(f"Warning:{rxn_id} not found or ATP is not a reactant.")

In [6]:
amac_model_strict_atp.reactions.rxn01517_c0

Reaction identifier,rxn01517_c0
Name,ATP:dUMP phosphotransferase [c0]
Memory address,0x158c26dd0
Stoichiometry,cpd00002_c0 + cpd00067_c0 + cpd00299_c0 --> cpd00008_c0 + cpd00978_c0 ATP + H+ + dUMP --> ADP + dUDP
GPR,WP_049588748.1
Lower bound,0
Upper bound,1000.0


In [ ]:
amac_model_strict_atp.metabolites.cpd00002_c0

Metabolite identifier,cpd00002_c0
Name,ATP
Memory address,0x159086390
Formula,C10H13N5O13P3
Compartment,c0
In 181 reaction(s),"rxn00695_c0, rxn05148_c0, rxn10254_c0, rxn02314_c0, rxn01219_c0, rxn00303_c0, rxn00117_c0, rxn03908_c0, rxn01513_c0, rxn09240_c0, rxn03141_c0, rxn02937_c0, rxn05251_c0, rxn00104_c0, rxn08298_c0,..."


In [8]:
amac_model_strict_atp.add_boundary(amac_model_strict_atp.metabolites.cpd00002_c0, type="demand")

Reaction identifier,DM_cpd00002_c0
Name,ATP demand
Memory address,0x12c0a0e10
Stoichiometry,cpd00002_c0 --> ATP -->
GPR,
Lower bound,0
Upper bound,1000.0


In [9]:
amac_basal_media = {
    "EX_cpd00058_e0": 1000,  # Cu2+_e0
    "EX_cpd00007_e0": 20,  # O2_e0
    "EX_cpd00971_e0": 1000,  # Na+_e0
    "EX_cpd00063_e0": 1000,  # Ca2+_e0
    "EX_cpd00048_e0": 1000,  # Sulfate_e0
    "EX_cpd10516_e0": 1000,  # fe3_e0
    "EX_cpd00254_e0": 1000,  # Mg_e0
    "EX_cpd00009_e0": 1000,  # Phosphate_e0
    "EX_cpd00205_e0": 1000,  # K+_e0
    "EX_cpd00099_e0": 1000,  # Cl-_e0
    "EX_cpd00030_e0": 1000,  # Mn2+_e0
    "EX_cpd00001_e0": 1000,  # H2O_e0
    "EX_cpd00034_e0": 1000,  # Zn2+_e0
    "EX_cpd00149_e0": 1000,  # Co2+_e0
}
glc_media = amac_basal_media.copy()
glc_media["EX_cpd00027_e0"] = 10  # Glucose_e0

In [10]:
amac_model_strict_atp.medium = glc_media

In [11]:
amac_model_strict_atp.objective = "DM_cpd00002_c0"

In [12]:
amac_model_strict_atp.optimize()

,fluxes,reduced_costs
rxn02201_c0,0.0,3.363085e-32
rxn00351_c0,0.0,2.209231e-16
rxn07431_c0,0.0,1.747864e-16
rxn00836_c0,0.0,1.154980e-15
rxn00423_c0,0.0,-5.548513e-15
...,...,...
EX_cpd00085_e0,0.0,-4.000000e-01
EX_cpd00039_e0,0.0,0.000000e+00
EX_cpd00054_e0,0.0,-4.000000e-01
rxn15341_c0,0.0,6.111759e-17


In [13]:
amac_model_strict_atp.medium

{'EX_cpd00048_e0': 1000,
 'EX_cpd00254_e0': 1000,
 'EX_cpd00001_e0': 1000,
 'EX_cpd00063_e0': 1000,
 'EX_cpd00034_e0': 1000,
 'EX_cpd00099_e0': 1000,
 'EX_cpd00205_e0': 1000,
 'EX_cpd10516_e0': 1000,
 'EX_cpd00058_e0': 1000,
 'EX_cpd00009_e0': 1000,
 'EX_cpd00027_e0': 10,
 'EX_cpd00007_e0': 20,
 'EX_cpd00030_e0': 1000,
 'EX_cpd00971_e0': 1000,
 'EX_cpd00149_e0': 1000}

In [15]:
print(amac_model_strict_atp.objective)

Maximize
1.0*DM_cpd00002_c0 - 1.0*DM_cpd00002_c0_reverse_47323
